# Stefan Problem: Enthalpy Method

We solve the two-phase Stefan problem in nondimensional variables. Instead of following the front explicitly, we write it in terms of the enthalpy $H$, which is the sensible heat plus the latent heat,
$$
H_t = \Delta \beta(H), \qquad \beta(H) = \alpha_s \min(H,0) + \max(H - H_m, 0), \qquad H_m = 1/\mathrm{St}.
$$
Here $u = \beta(H)$ is the Kirchhoff potential ($u = k\theta$ in each phase) and $\alpha_s = k_s/c_s$ is the diffusivity of the solid relative to the liquid. The solid is $\{H < 0\}$, the liquid is $\{H > H_m\}$, and in between $\theta = 0$ while the latent heat is absorbed. The front is never computed explicitly; it is recovered as the level set $\{f = 1/2\}$ of the liquid fraction $f = \min(1, \max(H/H_m, 0))$.

# Enthalpy Solver

$H$ lives on the cells of a uniform mesh of size $h$, and $u = \beta(H)$ is stored with an extra layer of ghost cells to impose the boundary conditions. Each step is explicit,
$$
H^{n+1} = H^n + \Delta t\, \Delta_h \beta(H^n),
$$
where $\Delta_h$ is the $(2d+1)$-point Laplacian. For a Dirichlet condition the ghost cell is chosen so that the mean of the ghost and the inner cell is the prescribed value on the face; for a Neumann condition the ghost cell gives the prescribed flux.

Since $\beta$ is nondecreasing and Lipschitz with constant $L_\beta = \max(1, \alpha_s)$, the scheme is monotone if $\Delta t \le h^2/(L_\beta\,\mathrm{diag})$, where $\mathrm{diag} = 2d$ plus one for each axis with a Dirichlet face. This is the property behind the standard convergence results for the enthalpy method. The scheme is also conservative; `boundary_energy` accumulates the heat that enters through the boundary, so the total enthalpy changes exactly by that amount.

# Exact Solution (Neumann)

To test the solver we use the Neumann solution. On $x > 0$ the solid starts at $\theta = -\theta_s$ and we fix $\theta = 1$ at $x = 0$. The front is $s(t) = 2\lambda\sqrt{t}$, with
$$
\theta = 1 - \frac{\mathrm{erf}(x/2\sqrt{t})}{\mathrm{erf}(\lambda)} \quad (x < s(t)), \qquad
\theta = -\theta_s + \theta_s\,\frac{\mathrm{erfc}(x/2\sqrt{\alpha_s t})}{\mathrm{erfc}(\nu\lambda)} \quad (x > s(t)), \qquad \nu = 1/\sqrt{\alpha_s},
$$
and $\lambda$ is the root of the Stefan condition at the front,
$$
\lambda\sqrt{\pi} - \mathrm{St}\,\frac{e^{-\lambda^2}}{\mathrm{erf}(\lambda)} + \frac{r_c\theta_s\mathrm{St}}{\nu\,\mathrm{erfcx}(\nu\lambda)} = 0.
$$

In [1]:
from enthalpy import *
from exact_solutions import neumann_lambda, neumann_theta, cell_average

# Verification

We start at $t_0 = 0.1$ from the cell averages of the exact enthalpy and run up to $T = 1$ on $[0, 8]$, with $\theta = 1$ at $x = 0$ and an insulated wall at $x = 8$ (far enough to approximate the half line). We compare the temperature in $L^1$ and $L^\infty$, the position of the front (the liquid volume against $s(T)$), and the energy balance, which should be zero up to rounding.

In [2]:
mat = Material(St=1.0, rc=1.0, rk=2.0)
theta_s, L, t0, T = 0.5, 8.0, 0.1, 1.0      # L large enough to approximate the half line
lam = neumann_lambda(mat, theta_s)
s0 = 2.0 * lam * np.sqrt(t0)

def H_exact_t0(x):
    th, _ = neumann_theta(x, t0, mat, theta_s, lam)
    return H_from_theta(th, mat, liquid=x < s0)

print(f"lambda = {lam:.6f}\n")
print(f"{'N':>6} {'L1 error':>10} {'Linf error':>11} {'front error':>12} {'energy balance':>15}")
for N in [200, 400, 800, 1600]:
    h = L / N
    x = (np.arange(N) + 0.5) * h
    # cell averages of H (not of theta) keep the latent heat of the cell cut by the front
    solver = EnthalpySolver((N,), h, mat, [(Dirichlet(1.0), Neumann())], cell_average(H_exact_t0, N, h), t0=t0)
    E0 = solver.total_enthalpy()
    solver.run(T)

    th_ex, s = neumann_theta(x, T, mat, theta_s, lam)
    err = np.abs(solver.theta - th_ex)
    balance = solver.total_enthalpy() - E0 - solver.boundary_energy
    print(f"{N:>6} {err.sum() * h:>10.2e} {err.max():>11.2e} {abs(solver.liquid_volume() - s):>12.2e} {balance:>15.1e}")

lambda = 0.441989

     N   L1 error  Linf error  front error  energy balance
   200   2.78e-03    1.15e-02     2.52e-03         2.2e-15
   400   1.07e-03    5.37e-03     1.25e-03        -6.7e-16
   800   3.53e-04    1.22e-03     4.76e-04         1.2e-14
  1600   1.46e-04    1.47e-03     4.26e-05         1.1e-13


# Visualization

We store each simulation as a sequence of snapshots and animate it afterwards, so the solver runs without callbacks. Since the fronts move like $\sqrt{t}$, we take the frames at $t_k = T (k/n)^p$ with $p \geq 2$; this way the early stage, where the dendritic structure changes fast, gets more frames.

In 1d and 2d the solid is light blue (almost white) and the liquid is Minecraft water blue. In 3d the liquid is transparent and the solid is Minecraft ice blue.

In [3]:
from visualization import *

## 1d: layers of ice and water

Random layers of ice and water at different temperatures, between a hot wall ($\theta = 1$) and a cold wall ($\theta = -1$). The thin layers disappear first, and at the end only one front between the walls remains. Below the temperature we show the space-time diagram of the phases.

In [4]:
mat = Material(St=0.7, rc=1.0, rk=2.0)
N = 800
h = 1.0 / N
x = (np.arange(N) + 0.5) * h

rng = np.random.default_rng(3)
edges = np.cumsum(np.r_[0.08, 0.92 * rng.dirichlet(np.full(16, 1.2))])
layer = np.searchsorted(edges, x)
solid = layer % 2 == 1
theta0 = np.where(solid, -rng.uniform(0.3, 1.0, len(edges) + 1)[layer],
                          rng.uniform(0.05, 0.4, len(edges) + 1)[layer])

solver = EnthalpySolver((N,), h, mat, [(Dirichlet(1.0), Dirichlet(-1.0))], H_from_theta(theta0, mat, liquid=~solid))
times, F1, TH1 = record(solver, t_end=0.12, n_frames=80, power=2.5)

In [ ]:
ani_1d = animate_1d(x, times, F1, TH1)
HTML(ani_1d.to_jshtml())
# ani_1d.save("stefan_1d.mp4", fps=15, dpi=120)

## 2d: snowflake

A cold snowflake ($\theta = -1.5$) in water at $\theta = 0.1$. The bottom wall is cold with a modulated temperature, the top wall is warm and the lateral walls are insulated. The water between the branches freezes and the tips melt, so the crystal rounds up before it merges with the ice that grows from the bottom. The thin lines are isotherms.

In [6]:
mat = Material(St=0.5, rc=1.0, rk=2.0)
N = 200
h = 1.0 / N
x = (np.arange(N) + 0.5) * h
X, Y = np.meshgrid(x, x, indexing="ij")

H0 = H_from_phase(snowflake_2d(X, Y), h, mat, theta_s=-1.5, theta_l=0.1)
bcs = [(Neumann(), Neumann()),
       (Dirichlet(lambda t, x, y: -1.5 * (1 + 0.3 * np.cos(6 * np.pi * x))), Dirichlet(0.6))]
solver = EnthalpySolver((N, N), h, mat, bcs, H0)
times, F2, TH2 = record(solver, t_end=0.15, n_frames=70, power=2.5)       # ~20 s

In [ ]:
ani_2d = animate_2d(times, F2, TH2)
HTML(ani_2d.to_jshtml())
# ani_2d.save("stefan_2d.mp4", fps=15, dpi=120)

## 3d: cubic dendrite

A cold dendrite ($\theta = -2$) in water at $\theta = 0.15$, in an insulated box. The side branches merge and the tips melt, so the crystal becomes more compact.

In [8]:
mat = Material(St=0.5, rc=1.0, rk=2.0)
N = 72
h = 1.0 / N
x = (np.arange(N) + 0.5) * h
X, Y, Z = np.meshgrid(x, x, x, indexing="ij")

H0 = H_from_phase(dendrite_3d(X, Y, Z), h, mat, theta_s=-2.0, theta_l=0.15)
solver = EnthalpySolver((N, N, N), h, mat, [(Neumann(), Neumann())] * 3, H0)
times, F3, _ = record(solver, t_end=0.08, n_frames=50, power=2.5, keep_theta=False)   # ~30 s

In [ ]:
ani_3d = animate_3d(times, F3, h)
HTML(ani_3d.to_jshtml())
# ani_3d.save("stefan_3d.mp4", fps=12, dpi=120)